# 03 — TensorFlow / Keras Training

End-to-end bacteria classification with TensorFlow 2.x + Keras:
1. Load data with `image_dataset_from_directory`
2. Build an EfficientNetV2-S feature extractor (ImageNet weights)
3. Two-phase training: head-only warm-up → full fine-tune
4. Evaluate — accuracy, confusion matrix, classification report


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

DATA_ROOT   = '../data/bacteria'
IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
NUM_CLASSES = 33
SEED        = 42

## 1. Data pipeline

In [ ]:
# Stratified split via sklearn, then build tf.data pipelines
import pathlib
from sklearn.model_selection import train_test_split

root = pathlib.Path(DATA_ROOT)
all_images = sorted(root.glob('*/*'))
all_labels_str = [p.parent.name for p in all_images]
classes = sorted(set(all_labels_str))
cls2idx = {c: i for i, c in enumerate(classes)}
all_labels = [cls2idx[l] for l in all_labels_str]
all_images_str = [str(p) for p in all_images]

train_idx, temp_idx = train_test_split(range(len(all_images_str)), test_size=0.30,
                                       stratify=all_labels, random_state=SEED)
temp_labels = [all_labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50,
                                     stratify=temp_labels, random_state=SEED)

print(f'Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}')

def build_dataset(indices, augment=False):
    imgs   = [all_images_str[i] for i in indices]
    labels = [all_labels[i]     for i in indices]
    ds = tf.data.Dataset.from_tensor_slices((imgs, labels))
    def parse(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMAGE_SIZE)
        img = tf.cast(img, tf.float32) / 255.0
        # ImageNet normalisation
        mean = tf.constant([0.485, 0.456, 0.406])
        std  = tf.constant([0.229, 0.224, 0.225])
        img  = (img - mean) / std
        if augment:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_flip_up_down(img)
            img = tf.image.random_brightness(img, max_delta=0.2)
            img = tf.image.random_contrast(img, 0.8, 1.2)
        return img, label
    ds = ds.map(parse, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.shuffle(1024, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = build_dataset(train_idx, augment=True)
val_ds   = build_dataset(val_idx)
test_ds  = build_dataset(test_idx)

## 2. Model — EfficientNetV2-S backbone

In [ ]:
base = keras.applications.EfficientNetV2S(
    weights='imagenet', include_top=False,
    input_shape=(*IMAGE_SIZE, 3)
)
base.trainable = False  # Phase 1: head only

inputs = keras.Input(shape=(*IMAGE_SIZE, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES)(x)
model = Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=3e-3, weight_decay=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
model.summary(line_length=80)

## 3. Phase 1 — head warm-up (10 epochs)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint('/tmp/best_bacteria_tf.keras',
                                    save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True,
                                  monitor='val_accuracy'),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6,
                                      monitor='val_loss'),
]

hist1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=10, callbacks=callbacks, verbose=1
)

## 4. Phase 2 — full fine-tune (20 more epochs)

In [ ]:
base.trainable = True
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
hist2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=20, callbacks=callbacks, verbose=1
)

## 5. Evaluation

In [ ]:
model.load_weights('/tmp/best_bacteria_tf.keras')

y_true, y_pred = [], []
for imgs, labels in test_ds:
    logits = model(imgs, training=False)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(tf.argmax(logits, 1).numpy().tolist())

acc = np.mean(np.array(y_true) == np.array(y_pred)) * 100
print(f'Test accuracy: {acc:.2f}%')
print(classification_report(y_true, y_pred,
                             target_names=[c.replace('_',' ') for c in classes]))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[c.replace('_',' ') for c in classes],
            yticklabels=[c.replace('_',' ') for c in classes], ax=ax)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
ax.set_title('Confusion Matrix (Normalised)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()